In [ ]:
import sys 
sys.path.insert(0, '..')

from app.config import PRICES, cost_usd
print('модели в таблице цен:', list(PRICES))
RUN_PAID = False

In [ ]:
from types import SimpleNamespace

fake_usage = SimpleNamespace(input_tokens=22, output_tokens=49)
for model in PRICES:
    print(model, f"${cost_usd(fake_usage, model):.6f}")

In [ ]:
import json
from pathlib import Path

gold = {
    "trial_name": "OlympiA",
    "citation": "Tutt ANJ et al. N Engl J Med 2021",
    "source": "PubMed PMID 34081848; полный текст PMC9126186",
    "on_slide": {"trial_name": "Olympiad trial"},
    "endpoints": {
        "IDFS": {
            "rate_3y_olaparib": 85.9, "rate_3y_placebo": 77.1,
            "diff": 8.8, "diff_ci_low": 4.5, "diff_ci_high": 13.0,
            "hr": 0.58, "hr_ci_level": 99.5, "hr_ci_low": 0.41, "hr_ci_high": 0.82,
            "p": "<0.001",
            "events_olaparib": 106, "events_placebo": 178,
        },
        "DDFS": {
            "rate_3y_olaparib": 87.5, "rate_3y_placebo": 80.4,
            "diff": 7.1, "diff_ci_low": 3.0, "diff_ci_high": 11.1,
            "hr": 0.57, "hr_ci_level": 99.5, "hr_ci_low": 0.39, "hr_ci_high": 0.83,
            "p": "<0.001",
        },
    },
}

path = Path("../tests/gold/olympia_2021.json")
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(gold, ensure_ascii=False, indent=2), encoding="utf-8")

checked = 2 + sum(len(fields) for fields in gold["endpoints"].values())
print("записано:", path, "· проверяемых значений:", checked)

In [ ]:
from pathlib import Path
from PIL import Image

slide_path = sorted(Path("../tests/slides/real").glob("*.png"))[0]
slide = Image.open(slide_path)
print("размер:", slide.size, "·", round(slide.width * slide.height / 1_000_000, 2), "Мп")
slide


In [ ]:
scale = (1_150_000 / (slide.width * slide.height)) ** 0.5
model_view = slide.resize((int(slide.width * scale), int(slide.height * scale)))
print("так примерно видит модель:", model_view.size)
model_view

In [ ]:
forest = slide.crop((860, 540, 1942, 990))
print("размер участка:", forest.size)
forest

In [ ]:
def make_tiles(img, cols=2, rows=2, overlap=0.15):
    """Нарезать картинку на cols × rows частей; соседние части перекрываются на overlap."""
    tile_w = int(img.width / cols * (1 + overlap))
    tile_h = int(img.height / rows * (1 + overlap))
    tiles = []
    for r in range(rows):
        for c in range(cols):
            left = min(int(c * img.width / cols), img.width - tile_w)
            top = min(int(r * img.height / rows), img.height - tile_h)
            tiles.append(img.crop((left, top, left + tile_w, top + tile_h)))
    return tiles


tiles = make_tiles(slide)
for i, tile in enumerate(tiles):
    print(i, tile.size, round(tile.width * tile.height / 1_000_000, 2), "Мп")
tiles[2]

In [ ]:
import base64
import io
import anthropic
from dotenv import load_dotenv

load_dotenv("../.env")
client = anthropic.Anthropic()


def image_to_base64(img):
    """Картинку Pillow → строка base64 в формате PNG (без записи на диск)."""
    buffer = io.BytesIO()
    img.save(buffer, format="PNG")
    return base64.standard_b64encode(buffer.getvalue()).decode("utf-8")


encoded = image_to_base64(tiles[3])
print("часть 3 в base64:", len(encoded), "символов · начало:", encoded[:12])

In [ ]:
import time


def read_slide(images, model, instruction):
    """Отправить картинки и вопрос модели; вернуть текст ответа, расход и время."""
    if not RUN_PAID:
        raise RuntimeError("платный вызов выключен: поставь RUN_PAID = True в первой ячейке")
    content = []
    for img in images:
        content.append({
            "type": "image",
            "source": {"type": "base64", "media_type": "image/png", "data": image_to_base64(img)},
        })
    content.append({"type": "text", "text": instruction})

    t0 = time.perf_counter()
    response = client.messages.create(
        model=model,
        max_tokens=4096,
        messages=[{"role": "user", "content": content}],
    )
    seconds = time.perf_counter() - t0

    text = "".join(block.text for block in response.content if block.type == "text")
    return {
        "model": model,
        "images": len(images),
        "text": text,
        "input_tokens": response.usage.input_tokens,
        "output_tokens": response.usage.output_tokens,
        "cost_usd": cost_usd(response.usage, model),
        "seconds": round(seconds, 1),
        "stop_reason": response.stop_reason,
    }

In [ ]:
import json
from datetime import datetime
from pathlib import Path


def save_run(run, name):
    """Сохранить результат прогона в файл data/runs/<дата>_<имя>.json; вернуть путь к файлу."""
    stamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    folder = Path("../data/runs")
    folder.mkdir(parents=True, exist_ok=True)
    path = folder / f"{stamp}_{name}.json"
    path.write_text(json.dumps(run, ensure_ascii=False, indent=2), encoding="utf-8")
    return path

In [ ]:
test_path = save_run({"model": "проба", "text": "проверка записи"}, "test")
print("записано:", test_path)
print(test_path.read_text(encoding="utf-8"))

In [ ]:
instruction = (
    "Это слайд доклада. Если картинок несколько — это части одного слайда, соседние части перекрываются. "
    "Перечисли по пунктам, что написано на слайде: заголовок, препараты, названия исследований, "
    "все числа как на слайде. Ничего не добавляй от себя."
)

run_whole = read_slide([slide], "claude-haiku-4-5", instruction)
print(save_run(run_whole, "haiku_whole_free"), f"· ${run_whole['cost_usd']:.4f} · {run_whole['seconds']} с · {run_whole['stop_reason']}")

run_tiles = read_slide(tiles, "claude-haiku-4-5", instruction)
print(save_run(run_tiles, "haiku_tiles_free"), f"· ${run_tiles['cost_usd']:.4f} · {run_tiles['seconds']} с · {run_tiles['stop_reason']}")

In [ ]:
print(run_tiles["text"])

In [ ]:
from pydantic import BaseModel


class Patient(BaseModel):
    name: str                # строка — обязательно
    age: int                 # целое число — обязательно
    weight_kg: float | None  # число или None («не измерено»)


ok = Patient(name="Иванов", age=64, weight_kg=None)
print(ok)

bad = Patient(name="Петров", age="шестьдесят", weight_kg=80.5)

In [ ]:
from pydantic import BaseModel, Field


class Arm(BaseModel):
    name: str = Field(description="название группы, как на слайде, например Olaparib")
    events: int | None = Field(description="число событий в группе, если подписано на слайде; иначе null")
    rate_3y: float | None = Field(description="значение кривой этой группы на 36-м месяце, %, как подписано; иначе null")


class Endpoint(BaseModel):
    name: str = Field(description="конечная точка, как на слайде")
    arms: list[Arm]
    diff: float | None
    diff_ci_level: float | None
    diff_ci_low: float | None
    diff_ci_high: float | None
    hr: float | None
    hr_ci_level: float | None = Field(description="уровень доверительного интервала для HR, как НАПЕЧАТАН на слайде: 95, 99, 99.5…")
    hr_ci_low: float | None
    hr_ci_high: float | None
    p: str | None = Field(description="p-значение строкой со знаком, как на слайде: '<0.001', '=0.02'")


class SlideReading(BaseModel):
    title: str | None
    trial_name: str | None
    citation: str | None
    endpoints: list[Endpoint]
    uncertain: list[str] = Field(description="что прочитано неуверенно или не читается — каждое отдельной строкой")


print("бланки внутри:", list(SlideReading.model_json_schema()["$defs"]))

In [ ]:
import time


def read_slide_structured(images, model, instruction):
    """Отправить картинки и вопрос модели; вернуть текст ответа, расход и время."""
    if not RUN_PAID:
        raise RuntimeError("платный вызов выключен: поставь RUN_PAID = True в первой ячейке")
    content = []
    for img in images:
        content.append({
            "type": "image",
            "source": {"type": "base64", "media_type": "image/png", "data": image_to_base64(img)},
        })
    content.append({"type": "text", "text": instruction})

    t0 = time.perf_counter()
    response = client.messages.parse(
        model=model,
        max_tokens=16000,
        messages=[{"role": "user", "content": content}],
        output_format=SlideReading,
    )
    seconds = time.perf_counter() - t0

    return {
        "model": model,
        "images": len(images),
        "reading": response.parsed_output.model_dump() if response.parsed_output else None,
        "input_tokens": response.usage.input_tokens,
        "output_tokens": response.usage.output_tokens,
        "cost_usd": cost_usd(response.usage, model),
        "seconds": round(seconds, 1),
        "stop_reason": response.stop_reason,
    }

In [ ]:
instruction_form = (
    "Это слайд доклада. Если картинок несколько — это части одного слайда, соседние части перекрываются. "
    "Заполни бланк строго по тому, что напечатано на слайде. Ничего не додумывай: если число не читается "
    "или ты не уверен — поставь null и добавь пункт в uncertain."
)

run = read_slide_structured(tiles, "claude-haiku-4-5", instruction_form)
print(save_run(run, "haiku_tiles_form"), f"· ${run['cost_usd']:.4f} · {run['seconds']} с · {run['stop_reason']}")

for ep in run["reading"]["endpoints"]:
    print(ep["name"], "· HR", ep["hr"], f"({ep['hr_ci_level']}% ДИ {ep['hr_ci_low']}–{ep['hr_ci_high']})", "· P", ep["p"])
    for arm in ep["arms"]:
        print("    ", arm["name"], "· событий", arm["events"], "· 3 года", arm["rate_3y"])
print("сомнения модели:", run["reading"]["uncertain"])

In [ ]:
from PIL import ImageOps

phone_path = Path("../tests/slides/real/olympia_phone_1280.jpg")
phone = ImageOps.exif_transpose(Image.open(phone_path)).convert("RGB")
phone_slide = phone.crop((40, 190, 1165, 830))       # только слайд: без браузера и окон докладчиков
print("фото:", phone.size, "· слайд:", phone_slide.size)

run_phone = read_slide_structured([phone_slide], "claude-haiku-4-5", instruction_form)
print(save_run(run_phone, "haiku_phone1280_form"), f"· ${run_phone['cost_usd']:.4f} · {run_phone['stop_reason']}")

for ep in run_phone["reading"]["endpoints"]:
    print(ep["name"], "· HR", ep["hr"], f"({ep['hr_ci_level']}% ДИ {ep['hr_ci_low']}–{ep['hr_ci_high']})", "· P", ep["p"])
    for arm in ep["arms"]:
        print("    ", arm["name"], "· событий", arm["events"], "· 3 года", arm["rate_3y"])
print("сомнения модели:", run_phone["reading"]["uncertain"])

In [ ]:
CODES = {"invasive": "IDFS", "distant": "DDFS"}   # первое слово названия → код, как в эталоне


def flatten(reading):
    """Бланк модели → плоский словарь {"адрес": значение}, например {"IDFS.hr": 0.58}."""
    flat = {"trial_name": reading["trial_name"], "citation": reading["citation"], "не сверено": []}
    for ep in reading["endpoints"]:
        code = CODES.get(ep["name"].split()[0].lower())
        if code is None:
            continue                                  # конечной точки нет в эталоне — пропускаем
        if f"{code}.hr" in flat:
            flat["не сверено"].append(ep["name"])     # второй результат того же кода — не затираем первый
            continue
        for field, value in ep.items():
            if field in ("name", "arms"):
                continue                              # не числа — разбираем отдельно ниже
            flat[f"{code}.{field}"] = value
        for arm in ep["arms"]:
            arm_name = arm["name"].lower()            # "Olaparib" → "olaparib", как в эталоне
            flat[f"{code}.rate_3y_{arm_name}"] = arm["rate_3y"]
            flat[f"{code}.events_{arm_name}"] = arm["events"]
    return flat


saved = json.loads(Path("../data/runs/2026-09-19_13-11-43_haiku_tiles_form.json").read_text(encoding="utf-8"))
flat = flatten(saved["reading"])
print("адресов:", len(flat))
for address, value in flat.items():
    print(f"{address:24} {value}")

In [ ]:
for code, fields in gold["endpoints"].items():
    print(code, "→", fields)
for code, fields in gold["endpoints"].items():
    for field, value in fields.items():
        print(f"{code}.{field}", value)

In [ ]:
def flatten_gold(gold):
    """Эталон → плоский словарь с теми же адресами, что у flatten."""
    flat = {"trial_name": gold["on_slide"]["trial_name"], "citation": gold["citation"]}
    for code, fields in gold["endpoints"].items():
        for field, value in fields.items():
            flat[f"{code}.{field}"] = value
    return flat


flat_gold = flatten_gold(gold)
print("адресов:", len(flat_gold))
for address, value in flat_gold.items():
    print(f"{address:24} {value}")

In [ ]:
def normalize(text):
    """Строку → вид для сравнения: строчные буквы, без пробелов и точек. Знак «<» остаётся."""
    return text.lower().replace(" ", "").replace(".", "")


def same(a, b):
    """Совпадают ли два значения: числа — точно, строки — после нормализации."""
    if isinstance(a, str) or isinstance(b, str):
        return normalize(str(a)) == normalize(str(b))
    return a == b


print(same(0.58, 0.58), same(13, 13.0), same("Tutt ANJ et al. N Engl J Med. 2021", "Tutt ANJ et al. N Engl J Med 2021"))
print(same("<0.001", "=0.001"), same(0.81, 0.83))

In [ ]:
def score(reading, gold):
    """Сверить бланк модели с эталоном; вернуть счёт по четырём графам и список расхождений."""
    flat = flatten(reading)
    doubts = " ".join(reading["uncertain"]).lower()      # все сомнения модели одной строкой
    counts = {"верно": 0, "null": 0, "с пометкой": 0, "без пометки": 0}
    counts["опознано"] = normalize(gold["trial_name"]) in normalize(reading["trial_name"] or "")
    misses = []
    for address, expected in flatten_gold(gold).items():
        if address in ("trial_name", "citation"):
            continue                                      # опознание считаем отдельно от чисел (решение № 30)
        got = flat.get(address)
        if got is None:
            counts["null"] += 1                           # не прочитано — честное «нет данных»
        elif same(got, expected):
            counts["верно"] += 1
        elif str(got).lower() in doubts:
            counts["с пометкой"] += 1                     # ошиблась, но сама об этом сказала
            misses.append((address, got, expected, "с пометкой"))
        else:
            counts["без пометки"] += 1                    # ошиблась молча — самое опасное
            misses.append((address, got, expected, "без пометки"))
    return counts, misses


counts, misses = score(saved["reading"], gold)
print(counts)
for miss in misses:
    print("   ", miss)

In [ ]:
for path in sorted(Path("../data/runs").glob("*form*.json")):
    run = json.loads(path.read_text(encoding="utf-8"))
    counts, misses = score(run["reading"], gold)
    print(f"{path.name:48} {counts}")

In [ ]:
import copy

broken = copy.deepcopy(saved["reading"])          # полная копия — оригинал не трогаем
broken["endpoints"][0]["hr"] = 0.85               # IDFS: HR 0.58 → 0.85 (переставлены цифры)
broken["endpoints"][1]["hr_ci_level"] = 95.0      # DDFS: 99.5 % → 95 %, типичная ошибка модели

counts, misses = score(broken, gold)
print(counts)
for miss in misses:
    print("   ", miss)
print("оригинал цел:", saved["reading"]["endpoints"][0]["hr"])

In [ ]:
phone2000 = ImageOps.exif_transpose(Image.open("../tests/slides/real/olympia_phone_2000.jpg")).convert("RGB")
phone2000_slide = phone2000.crop((65, 300, 1815, 1290))      # только слайд, без браузера — проверено на глаз

variants = {
    "screen_whole": [slide],                    # снимок экрана целиком
    "screen_tiles": tiles,                      # снимок экрана нарезкой (сделано выше)
    "phone_whole": [phone2000_slide],           # фото телефона целиком
    "phone_tiles": make_tiles(phone2000_slide), # фото телефона нарезкой
}
for name, images in variants.items():
    print(f"{name:13}", len(images), "карт. ·", images[0].size)

In [ ]:
models = ["claude-haiku-4-5", "claude-sonnet-5", "claude-opus-5"]
repeats = 2

for model in models:
    for variant, images in variants.items():
        for repeat in range(1, repeats + 1):
            name = f"m7_{model}_{variant}_{repeat}"
            run = read_slide_structured(images, model, instruction_form)
            total += run["cost_usd"]
            save_run(run, name)
            if run["reading"] is None:
                failed.append((name, run["stop_reason"]))
                print(f"{name:40} бланка нет · {run['stop_reason']}")
                continue
            counts, misses = score(run["reading"], gold)
            results.append({"model": model, "variant": variant, "repeat": repeat, **counts,
                            "cost_usd": run["cost_usd"], "seconds": run["seconds"], "misses": misses})
            print(f"{name:40} {counts} · ${run['cost_usd']:.4f} · {run['seconds']} с")

print(f"прогонов: {len(results) + len(failed)} · с бланком: {len(results)} · без бланка: {len(failed)} · расход: ${total:.4f}")

In [ ]:
MAX_SILENT = 0.02    # порог № 27: неверных чисел без пометки — не больше 2 %
MAX_NULL = 0.05      # решение № 30: «не прочитано» — не больше 5 %

print(f"{'прогон':44} {'верно':>5} {'null':>5} {'помет':>5} {'молча':>5}  опозн.  порог")
for path in sorted(Path("../data/runs").glob("*_m7_*.json")):
    run = json.loads(path.read_text(encoding="utf-8"))
    counts, misses = score(run["reading"], gold)
    numbers = counts["верно"] + counts["null"] + counts["с пометкой"] + counts["без пометки"]
    passed = (counts["опознано"]
              and counts["без пометки"] / numbers <= MAX_SILENT
              and counts["null"] / numbers <= MAX_NULL)
    print(f"{path.stem[20:]:44} {counts['верно']:>5} {counts['null']:>5} {counts['с пометкой']:>5} "
          f"{counts['без пометки']:>5}  {'да' if counts['опознано'] else 'НЕТ':6}  {'ПРОШЁЛ' if passed else '—'}")

In [ ]:
import re
from rapidocr import RapidOCR

ocr_engine = RapidOCR()      # модели уже лежат внутри пакета — ничего не скачивается

ocr_results = {}
for source, img in [("screen", slide.convert("RGB")), ("phone", phone2000_slide)]:
    t0 = time.perf_counter()
    ocr_results[source] = ocr_engine(img)
    print(source, "· строк:", len(ocr_results[source].txts), "·", round(time.perf_counter() - t0, 1), "с")

for text, conf in list(zip(ocr_results["screen"].txts, ocr_results["screen"].scores))[:12]:
    print(f"   {conf:.2f}  {text}")

In [ ]:
def ocr_read(result):
    """Результат OCR → все числа со слайда (множество) и весь текст одной нормализованной строкой."""
    numbers = set()
    for text in result.txts:
        for token in re.findall(r"\d+(?:[.,]\d+)?", text):
            numbers.add(float(token.replace(",", ".")))
    return {"numbers": numbers, "text": normalize(" ".join(result.txts))}


def confirmed(value, ocr):
    """Есть ли значение на слайде по данным OCR: число — среди чисел, строка — в тексте."""
    if isinstance(value, str):
        return normalize(value) in ocr["text"]
    return float(value) in ocr["numbers"]


ocr_by_source = {}
for source, result in ocr_results.items():
    ocr_by_source[source] = ocr_read(result)

for source, ocr in ocr_by_source.items():
    missing = []
    for address, value in flatten_gold(gold).items():
        if address in ("trial_name", "citation"):
            continue
        if not confirmed(value, ocr):
            missing.append(address)
    print(source, "· чисел эталона нашёл OCR:", 22 - len(missing), "из 22 · не нашёл:", missing)

In [ ]:
table = {}
for path in sorted(Path("../data/runs").glob("*_m7_*.json")):
    run = json.loads(path.read_text(encoding="utf-8"))
    source = "screen" if "_screen_" in path.name else "phone"
    flat = flatten(run["reading"])
    doubts = " ".join(run["reading"]["uncertain"]).lower()
    row = table.setdefault(run["model"], {"прогонов": 0, "молча": 0, "молча → поймал OCR": 0, "ложная тревога": 0})
    row["прогонов"] += 1
    for address, expected in flatten_gold(gold).items():
        got = flat.get(address)
        if address in ("trial_name", "citation") or got is None:
            continue
        seen = confirmed(got, ocr_by_source[source])
        if same(got, expected):
            if not seen:
                row["ложная тревога"] += 1        # верное число, но OCR его не нашёл — лишняя пометка
        elif str(got).lower() not in doubts:
            row["молча"] += 1                     # ошибка без пометки модели
            if not seen:
                row["молча → поймал OCR"] += 1    # OCR такого числа на слайде не видит — пометка

for model, row in table.items():
    print(f"{model:18}", row)

In [ ]:
def ocr_hint(result, img):
    """OCR → текст подсказки: каждая строка с координатами центра в % от ширины и высоты слайда."""
    lines = []
    for box, text in zip(result.boxes, result.txts):
        x = round(box[:, 0].mean() / img.width * 100)     # box — 4 угла рамки; [:, 0] — их x
        y = round(box[:, 1].mean() / img.height * 100)    # [:, 1] — их y
        lines.append((y, x, text))
    lines.sort()                                           # сверху вниз, затем слева направо
    out = []
    for y, x, text in lines:
        out.append(f"[x={x} y={y}] {text}")
    return "\n".join(out)


hints = {"screen": ocr_hint(ocr_results["screen"], slide), "phone": ocr_hint(ocr_results["phone"], phone2000_slide)}
print("строк в подсказке:", len(hints["screen"].splitlines()), "и", len(hints["phone"].splitlines()))
print(hints["screen"][:300])

In [ ]:
instruction_hint = (
    instruction_form + "\n\n"
    "Ниже — текст этого слайда, распознанный программой OCR, с координатами центра каждой строки "
    "в процентах: x — слева направо, y — сверху вниз (по целому слайду, даже если картинка нарезана). "
    "Цифры в OCR точнее, чем на картинке: значения чисел бери из OCR, а по картинке и координатам определяй, "
    "к какой панели, кривой и показателю относится число. Числа нет в OCR — читай с картинки и добавь пункт в uncertain.\n\n"
)

hint_results = []
hint_total = 0.0
for variant, images in variants.items():
    source = "screen" if variant.startswith("screen") else "phone"
    for repeat in range(1, 3):
        name = f"m8_hint_claude-haiku-4-5_{variant}_{repeat}"
        try:
            run = read_slide_structured(images, "claude-haiku-4-5", instruction_hint + hints[source])
        except Exception as error:
            print(f"{name:44} бланка нет · {type(error).__name__}")
            continue
        hint_total += run["cost_usd"]
        save_run(run, name)
        counts, misses = score(run["reading"], gold)
        hint_results.append((name, counts, misses))
        print(f"{name:44} {counts} · ${run['cost_usd']:.4f} · {run['seconds']} с")

print(f"прогонов с бланком: {len(hint_results)} · расход: ${hint_total:.4f}")

In [ ]:
for name, counts, misses in hint_results:
    numbers = counts["верно"] + counts["null"] + counts["с пометкой"] + counts["без пометки"]
    passed = (counts["опознано"]
              and counts["без пометки"] / numbers <= MAX_SILENT
              and counts["null"] / numbers <= MAX_NULL)
    print(f"{name:44} {'ПРОШЁЛ' if passed else '—'}")
    for miss in misses:
        print("     ", miss)